In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import psycopg2
from datetime import timedelta
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

In [2]:
class SqlWorker:
    def __init__(self):
        self.schema = 'temp'
        self.connection = psycopg2.connect(
            database='dom',
            user='dom_user',
            password='dom_password',
            host='localhost',
            port=5432,
        )

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.connection.close()

    def execute_query(self, query: str) -> str | None:
        """
        Wykonuje query i logguje wynik.
        """
        with self.connection.cursor() as cursor:
            cursor.execute(query)
            try:
                response = cursor.fetchall()
            except psycopg2.ProgrammingError:
                response = None

        self.connection.commit()
        return response

    def insert_data(self, temperature: float, humidity:int) -> None:
        self.execute_query(f"INSERT INTO {self.schema}.data(temperature, humidity) "
                           f"VALUES({temperature}, {humidity});")


In [ ]:
with SqlWorker() as sql:
    r = sql.execute_query('SELECT * FROM temp.data;')

df = pd.DataFrame(r, columns=['t', 'temp', 'humidity'])
df = pd.read_excel('data.xlsx')

df["t"] = df["t"].dt.tz_localize(None)
df['day'] = df['t'].dt.day
df['time'] = pd.to_datetime(df['t'].dt.strftime('2000-01-01 %H:%M:%S'))
df['dt'] = df['t'].diff().dt.seconds
df = df.sort_values(by=['day', 'time'], ascending=True)


df = df[
    (df['time'].dt.time >= pd.to_datetime("07:00").time()) &
    (df['time'].dt.time <= pd.to_datetime("12:00").time())
]

# df['temp'] = df['temp'] - 1.5
# df['humidity'] = df['humidity'] - 4

df.tail(5)

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Temperature', 'Humiditiy'])

days = [d for d in df['day'].unique() if d != 17]
colors = px.colors.qualitative.Plotly  # or any palette you like
color_map = {day: colors[i % len(colors)] for i, day in enumerate(days)}

for day in days:
    tmp = df.loc[df['day'] == day]
    color = color_map[day]

    fig.add_trace( go.Scatter(x=tmp['time'], y=tmp['temp'], name=str(day), mode="lines+markers",
    line=dict(color=color), marker=dict(symbol="cross", color=color, size=4)), 1, 1 )
    fig.add_trace(go.Scatter(x=tmp['time'], y=tmp['humidity'], name=str(day), showlegend=False, mode="lines+markers",
    line=dict(color=color), marker=dict(symbol="cross", color=color, size=4)), 2, 1 )

fig.add_hrect(y0=24-0.5, y1=24+0.5, fillcolor="red", opacity=0.2, row=1, col=1)
fig.add_hrect(y0=50-5, y1=50+5, fillcolor="red", opacity=0.2, row=2, col=1)

fig.update_layout(height=1000)
fig.show()

In [5]:
px.scatter(x = df.query('dt < 120')['t'], y = df.query('dt < 120')['dt'])

In [6]:
def abs_humidity(T_C, RH_percent):
    # Magnus formula for saturation vapor pressure (hPa)
    e_s = 6.112 * np.exp((17.67 * T_C) / (T_C + 243.5))
    # Actual vapor pressure
    e = e_s * RH_percent / 100.0
    # Absolute humidity (g/m﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿)
    AH = (216.7 * e) / (273.15 + T_C)
    return AH

In [7]:
df['abs_humidity'] = df.apply(lambda row: abs_humidity(row['temp'], row['humidity']), axis=1)

In [8]:
px.line(x=df['t'], y=df['abs_humidity'])

In [9]:
px.scatter(y=df['temp'], x=df['abs_humidity'])

In [10]:
def RC(R, C):
    return 10**3 / (C*R*5)

RC(1.95, 100)

1.0256410256410255

In [11]:
t_moje = 24.8;  h_moje = 38
t_zegar = 23.5; h_zegar = 34
t_mama = 25;    h_maszyna = 34

In [12]:
h_zegar = 73; h_moje = 68

In [13]:
h_moje = [38, 68, 67, 82, 85, 82]
h_zegar  = [34, 73, 78, 71, 72, 95]

t_moje = [None, None, 24.1]
t_zegar = [None, None, 21]

calib = pd.DataFrame({'h_moje': h_moje, 'h_zegar': h_zegar})#, 't_moje': t_moje, 't_zegar': t_zegar})

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, subplot_titles=['Temperature', 'Humiditiy'])


fig.add_trace(go.Scatter(x=calib['h_moje'], y=calib['h_zegar'], mode="markers", marker=dict(symbol="cross", size=4)), 1, 1 )
#fig.add_trace(go.Scatter(x=calib['t_moje'], y=calib['t_zegar'],showlegend=False, mode="markers", marker=dict(symbol="cross", size=4)), 2, 1 )

fig.update_layout(height=500)
fig.show()